<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/LDA_PHI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LDA

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 290 (delta 65), reused 4 (delta 4), pack-reused 175 (from 1)
Receiving objects: 100% (290/290), 15.09 MiB | 10.34 MiB/s, done.
Resolving deltas: 100% (116/116), done.


In [2]:
!wget -O CORPUS.csv "https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"

--2026-04-21 02:40:52--  https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 02:40:52--  https://virginia.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 02:40:52--  https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443.

In [3]:
import pandas as pd
CORPUS = pd.read_csv('CORPUS.csv', delimiter='|', index_col=[0,1,2,3])
LIB = pd.read_csv('LIB.csv', delimiter='|', index_col=0)

In [4]:
DOCS = CORPUS[CORPUS.pos.str.match(r'^NNS?$')]\
    .groupby(['Artist', 'Album', 'Title']).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

In [5]:
from sklearn.feature_extraction import text
my_stop_words = list(text.ENGLISH_STOP_WORDS)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
count_engine = CountVectorizer(max_df=.9, min_df=10, stop_words=my_stop_words)
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
COUNTS = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)

In [7]:
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation as LDA

n_topics = 20
max_iter = 100
n_top_terms = 9
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]
topic_engine = LDA(n_components=20, random_state=42)
topic_model = topic_engine.fit_transform(count_model)

## PHI

In [8]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.head()

term_str,act,adam,advice,affection,afraid,afternoon,age,ah,ahah,ahh,...,yesterday,yo,york,youd,youll,youre,youth,youve,yuh,yup
topic_id,,,,,,,,,,,,,,,,,,,,,
T00,0.050000,0.050000,0.050000,0.05,0.05,0.050000,0.050000,26.920135,0.05,0.05,...,0.050000,4.043830,10.575036,26.496026,42.270188,0.050000,0.05,9.728658,6.407605,0.050000
T01,7.607295,0.050000,0.050000,0.05,0.05,0.050000,0.050000,0.547858,0.05,0.05,...,0.050000,0.387412,0.729060,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,0.050000
T02,8.455275,0.050000,8.576520,0.05,0.05,0.050000,2.360304,8.912592,0.05,0.05,...,10.875341,6.417447,0.050000,3.630781,24.037208,36.220418,0.05,12.429305,1.108078,16.579982
T03,0.050000,27.411892,0.050000,0.05,0.05,0.050000,0.050000,0.050000,0.05,0.05,...,0.050000,0.050000,0.050000,0.050000,6.118775,214.122178,0.05,2.359839,0.050000,0.050000
T04,0.050000,0.050000,0.408442,0.05,1.05,1.232819,0.050000,0.050000,0.05,0.05,...,0.050000,0.050000,9.984235,23.002324,0.068388,37.010386,0.05,0.050000,0.050000,3.228586


In [9]:
PHI.to_csv("/content/DS5001-Final-Project/PHI.csv", sep="|", index=True)